# Kshetra Sense — leakage-free crop intelligence
This notebook mirrors the production workflow: raw-data splitting, training-only cross-validation, probability calibration, a single untouched test evaluation, OOD checks, model-agnostic explanations, and external validation. It deliberately makes no claim of geographic or seasonal validation.

In [ ]:
from pathlib import Path
import json, joblib, numpy as np, pandas as pd
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix, f1_score, log_loss, precision_score, recall_score, top_k_accuracy_score)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
FEATURES = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
TARGET, RANDOM_STATE = 'label', 42

## 1. Load and audit the source data
Fail fast on schema, missing values, and non-numeric inputs. The dataset is balanced, so synthetic oversampling is neither necessary nor appropriate here.

In [ ]:
df = pd.read_csv(ROOT / 'dataset' / 'Crop_recommendation.csv')
required = set(FEATURES + [TARGET])
assert required.issubset(df.columns)
assert not df[FEATURES + [TARGET]].isna().any().any()
assert all(pd.api.types.is_numeric_dtype(df[name]) for name in FEATURES)
audit = {'rows': len(df), 'classes': df[TARGET].nunique(), 'duplicates': int(df.duplicated().sum()), 'class_counts': df[TARGET].value_counts().to_dict()}
audit

## 2. Split raw values before preprocessing
The test partition is locked away immediately. Any scaling is placed inside an sklearn `Pipeline`, so every cross-validation fold learns preprocessing only from its own training fold.

In [ ]:
encoder = LabelEncoder()
X, y = df[FEATURES].copy(), encoder.fit_transform(df[TARGET])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.20, random_state=RANDOM_STATE, stratify=y)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
X_train.shape, X_test.shape

## 3. Select the model using training-only macro F1
Macro F1 weights every crop equally. Accuracy and balanced accuracy are tracked, while a most-frequent dummy classifier provides a real baseline.

In [ ]:
candidates = {
 'KNN': (Pipeline([('scale', StandardScaler()), ('model', KNeighborsClassifier())]), {'model__n_neighbors':[3,5,7], 'model__weights':['uniform','distance'], 'model__metric':['euclidean','manhattan']}),
 'Decision Tree': (Pipeline([('model', DecisionTreeClassifier(random_state=RANDOM_STATE))]), {'model__criterion':['gini','entropy'], 'model__max_depth':[None,10,20], 'model__min_samples_split':[2,5], 'model__min_samples_leaf':[1,2]}),
 'Random Forest': (Pipeline([('model', RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1))]), {'model__n_estimators':[100,250], 'model__max_depth':[None,10,20], 'model__min_samples_split':[2,5], 'model__min_samples_leaf':[1,2]}),
 'SVM': (Pipeline([('scale', StandardScaler()), ('model', SVC(random_state=RANDOM_STATE))]), {'model__C':[.1,1,10], 'model__gamma':['scale','auto'], 'model__kernel':['rbf','linear']})}
scoring = {'macro_f1':'f1_macro', 'balanced_accuracy':'balanced_accuracy', 'accuracy':'accuracy'}
searches, rows = {}, []
for name, (pipeline, grid) in candidates.items():
    search = GridSearchCV(pipeline, grid, scoring=scoring, refit='macro_f1', cv=cv, n_jobs=-1).fit(X_train, y_train)
    i, searches[name] = search.best_index_, search
    rows.append({'model':name, 'macro_f1':search.cv_results_['mean_test_macro_f1'][i], 'balanced_accuracy':search.cv_results_['mean_test_balanced_accuracy'][i], 'accuracy':search.cv_results_['mean_test_accuracy'][i]})
baseline = cross_validate(Pipeline([('model', DummyClassifier(strategy='most_frequent'))]), X_train, y_train, scoring=scoring, cv=cv)
rows.append({'model':'Dummy baseline', 'macro_f1':baseline['test_macro_f1'].mean(), 'balanced_accuracy':baseline['test_balanced_accuracy'].mean(), 'accuracy':baseline['test_accuracy'].mean()})
comparison = pd.DataFrame(rows).sort_values('macro_f1', ascending=False)
comparison.style.format({name:'{:.2%}' for name in ['macro_f1','balanced_accuracy','accuracy']})

## 4. Calibrate the champion, then open the test set once
Temperature scaling is fit through internal cross-validation on the training partition. The untouched test set is then used for final reporting—not model selection.

In [ ]:
champion_name = comparison.query("model != 'Dummy baseline'").iloc[0]['model']
champion = CalibratedClassifierCV(estimator=clone(searches[champion_name].best_estimator_), method='temperature', cv=cv, ensemble=False).fit(X_train, y_train)
probabilities = champion.predict_proba(X_test)
predictions = probabilities.argmax(axis=1)
labels = np.arange(len(encoder.classes_))
metrics = {
 'accuracy': accuracy_score(y_test, predictions),
 'balanced_accuracy': balanced_accuracy_score(y_test, predictions),
 'macro_precision': precision_score(y_test, predictions, average='macro'),
 'macro_recall': recall_score(y_test, predictions, average='macro'),
 'macro_f1': f1_score(y_test, predictions, average='macro'),
 'top_3_accuracy': top_k_accuracy_score(y_test, probabilities, k=3, labels=labels),
 'log_loss': log_loss(y_test, probabilities, labels=labels),
 'multiclass_brier': np.mean(np.sum((probabilities - np.eye(len(labels))[y_test])**2, axis=1))}
metrics

In [ ]:
report = pd.DataFrame(classification_report(y_test, predictions, target_names=encoder.classes_, output_dict=True)).T
matrix = pd.DataFrame(confusion_matrix(y_test, predictions), index=encoder.classes_, columns=encoder.classes_)
display(report, matrix)

## 5. Inspect probability calibration and model-agnostic importance
Reliability bins compare mean predicted probability with observed accuracy. Permutation importance reports test macro-F1 loss after shuffling one feature; it is not a causal claim.

In [ ]:
confidence, correct = probabilities.max(axis=1), predictions == y_test
edges, reliability, ece = np.linspace(0, 1, 11), [], 0.0
for lower, upper in zip(edges[:-1], edges[1:]):
    mask = (confidence >= lower) & (confidence < upper if upper < 1 else confidence <= upper)
    if mask.any():
        bin_conf, bin_acc = confidence[mask].mean(), correct[mask].mean()
        ece += mask.mean() * abs(bin_conf - bin_acc)
        reliability.append({'lower':lower, 'upper':upper, 'records':mask.sum(), 'confidence':bin_conf, 'accuracy':bin_acc})
pd.DataFrame(reliability), {'expected_calibration_error': ece}

In [ ]:
permutation = permutation_importance(champion, X_test, y_test, scoring='f1_macro', n_repeats=20, random_state=RANDOM_STATE, n_jobs=-1)
pd.DataFrame({'feature':FEATURES, 'mean_f1_decrease':permutation.importances_mean, 'std':permutation.importances_std}).sort_values('mean_f1_decrease', ascending=False)

## 6. Use the same guarded production inference
The deployable artifact verifies checksums, constrains features to the training schema, measures multivariate novelty, abstains outside support, and supplies local one-at-a-time sensitivity. Sensitivity is associative, not causal.

In [ ]:
import sys
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from ml_core import assess_ood, load_model_bundle, local_sensitivity, predict_ranked
bundle = load_model_bundle(ROOT / 'models')
field = [90, 42, 43, 21.0, 82, 6.5, 203]
assessment = assess_ood(field, bundle.ood_profile)
ranking = [] if assessment['status'] == 'abstain' else predict_ranked(bundle, field, top_k=5)
assessment, ranking, local_sensitivity(bundle, field)

## 7. Validate transportability with genuinely external labelled data
Do not call the internal holdout external validation. Collect a CSV from a different geography, season, farm, or measurement process with the same seven features plus `label`, then evaluate it without modifying the model.

In [ ]:
from ml_core import validate_external_dataframe
external_path = ROOT / 'external_validation.csv'
if external_path.exists():
    external_results = validate_external_dataframe(bundle, pd.read_csv(external_path))
    display({key:value for key, value in external_results.items() if key != 'predictions'})
else:
    print('Add external_validation.csv to run a genuine out-of-source evaluation.')

## Limitations and next evidence
The source data contains no farm ID, location, time, variety, yield, price, disease pressure, or intervention outcome. The model therefore ranks crop labels within this dataset's feature space; it does not prove agronomic suitability, profitability, or causal benefit. The highest-value next step is prospective, multi-region, season-aware data collection followed by group/time-based validation and post-deployment drift monitoring.